# 06_Evaluation

Dokumentasi fase Evaluation. Metode evaluasi performa dan benchmark model berdasarkan MAE, RMSE, MAPE, sMAPE, dan Directional Accuracy.

## Load comparison results and summary

Sel yang memuat semua file perbandingan hasil eksperimen (jika tersedia) dan menyajikan ringkasan metrik per komoditas. Jika file tidak ditemukan, jalankan `notebooks/05_Modeling_v2.ipynb` atau skrip `src/train.py` untuk menghasilkan `runs/*_compare.csv`.

In [48]:
from pathlib import Path
import pandas as pd
project_root = Path('..').resolve()
compare_paths = [
    project_root / 'runs' / 'recomputed_compare.csv',
    project_root / 'runs' / 'all_commodity_optuna_summary.csv',
    project_root / 'runs' / 'all_commodity_compare.csv',
]
compare_path = next((p for p in compare_paths if p.exists()), None)

if compare_path is None:
    print('No comparison CSV found. Expected one of:')
    for p in compare_paths:
        print(' -', p)
else:
    print('Loading comparison CSV from', compare_path)
    comp = pd.read_csv(compare_path)
    display(comp)

    if compare_path.name == 'all_commodity_final_compare.csv':
        comp = comp.copy()
        if 'mae_improvement_vs_baseline_multivariate' in comp.columns:
            comp['mae_improvement_vs_baseline_multivariate'] = comp['mae_improvement_vs_baseline_multivariate'].round(3)
            comp['rmse_improvement_vs_baseline_multivariate'] = comp['rmse_improvement_vs_baseline_multivariate'].round(3)
            comp['smape_improvement_vs_baseline_multivariate'] = comp['smape_improvement_vs_baseline_multivariate'].round(3)
            display(comp.sort_values('mae_improvement_vs_baseline_multivariate', ascending=False))
            print('')
            print('Insight summary:')
            if (comp['mae_improvement_vs_baseline_multivariate'] > 0).all():
                print('Optuna retrained multivariate models improved MAE for all commodities versus the earlier baseline multivariate runs.')
            else:
                degraded = comp[comp['mae_improvement_vs_baseline_multivariate'] < 0]['commodity'].tolist()
                print('Commodities with worse Optuna multivariate results than baseline multivariate:', degraded)
            tune_recs = comp[comp['mae_improvement_vs_baseline_multivariate'] < 0]['commodity'].tolist()
            if len(tune_recs) > 0:
                print('Recommend a second investigation/tuning pass for:', tune_recs)
        else:
            print('Final compare file is missing improvement columns; falling back to per-mode comparison logic.')
            if {'mode', 'commodity', 'mae', 'rmse', 'smape'}.issubset(comp.columns):
                uni = comp[comp['mode'] == 'univariate'].set_index('commodity')
                mv = comp[comp['mode'] == 'multivariate'].set_index('commodity')
                merged = uni.join(mv, lsuffix='_uni', rsuffix='_mv')
                merged = merged.dropna()
                merged['mae_delta'] = merged['mae_uni'] - merged['mae_mv']
                merged['rmse_delta'] = merged['rmse_uni'] - merged['rmse_mv']
                merged['smape_delta'] = merged['smape_uni'] - merged['smape_mv']
                summary = merged[['mae_uni','mae_mv','mae_delta','rmse_uni','rmse_mv','rmse_delta','smape_uni','smape_mv','smape_delta']].reset_index()
                summary = summary.sort_values('mae_delta', ascending=False)
                display(summary)
                tune_recs = summary[(summary['mae_delta'] < 0) | (summary['mae_delta'].abs() < 1.0)]['commodity'].tolist()
                print('Recommended commodities for hyperparameter tuning (multivariate worse or similar):', tune_recs)
            else:
                print('Unable to compute fallback summary from the current file structure.')
                tune_recs = []
    elif compare_path.name == 'all_commodity_optuna_summary.csv':
        print('This file contains final Optuna retrain metrics for each commodity. To compare against baseline multivariate results, load all_commodity_final_compare.csv.')
        tune_recs = []
    else:
        uni = comp[comp['mode'] == 'univariate'].set_index('commodity')
        mv = comp[comp['mode'] == 'multivariate'].set_index('commodity')
        merged = uni.join(mv, lsuffix='_uni', rsuffix='_mv')
        merged = merged.dropna()
        merged['mae_delta'] = merged['mae_uni'] - merged['mae_mv']
        merged['rmse_delta'] = merged['rmse_uni'] - merged['rmse_mv']
        merged['smape_delta'] = merged['smape_uni'] - merged['smape_mv']
        summary = merged[['mae_uni','mae_mv','mae_delta','rmse_uni','rmse_mv','rmse_delta','smape_uni','smape_mv','smape_delta']].reset_index()
        summary = summary.sort_values('mae_delta', ascending=False)
        display(summary)
        tune_recs = summary[(summary['mae_delta'] < 0) | (summary['mae_delta'].abs() < 1.0)]['commodity'].tolist()
        print('Recommended commodities for hyperparameter tuning (multivariate worse or similar):', tune_recs)
    globals()['tune_recs'] = tune_recs


Loading comparison CSV from /home/rna_13/DataScienceProject/dsproject_agriculturepredict/runs/recomputed_compare.csv


,commodity,mode,epochs,mae,rmse,mape,smape,directional_acc
0,Beras Premium,univariate,60,183.995,184.167,1.415,1.405,0.0
1,Beras Premium,multivariate,60,465.147,465.448,3.578,3.515,0.0
2,Beras Medium,univariate,60,70.425,71.493,0.563,0.562,0.0
3,Beras Medium,multivariate,60,44.176,45.179,0.353,0.353,0.0
4,Jagung Pipil Kering,univariate,60,91.250,93.407,2.340,2.369,0.0
5,Jagung Pipil Kering,multivariate,60,117.196,118.027,3.005,2.960,0.0
6,Kacang Hijau,univariate,60,9.844,10.925,0.076,0.076,0.0
7,Kacang Hijau,multivariate,60,3.378,4.004,0.026,0.026,0.0


,commodity,mae_uni,mae_mv,mae_delta,rmse_uni,rmse_mv,rmse_delta,smape_uni,smape_mv,smape_delta
1,Beras Medium,70.425,44.176,26.249,71.493,45.179,26.314,0.562,0.353,0.209
3,Kacang Hijau,9.844,3.378,6.466,10.925,4.004,6.921,0.076,0.026,0.050
2,Jagung Pipil Kering,91.250,117.196,-25.946,93.407,118.027,-24.620,2.369,2.960,-0.591
0,Beras Premium,183.995,465.147,-281.152,184.167,465.448,-281.281,1.405,3.515,-2.110


Recommended commodities for hyperparameter tuning (multivariate worse or similar): ['Jagung Pipil Kering', 'Beras Premium']


In [ ]:
# Jalankan baseline ulang untuk satu komoditas dari notebook jika diperlukan
run_baseline = True
if run_baseline:
    import subprocess, sys
    from pathlib import Path
    project_root = Path('..').resolve()
    baseline_cmd = [
        sys.executable,
        str(project_root / 'src' / 'train.py'),
        '--commodity', 'Beras Medium',
        '--mode', 'both',
        '--epochs', '60',
        '--device', 'cpu',
        '--log-csv', str(project_root / 'runs' / 'recomputed_metrics_Beras_Medium.csv'),
        '--compare-csv', str(project_root / 'runs' / 'recomputed_compare.csv'),
        '--early-stopping-patience', '5',
    ]
    print('Running baseline recompute for Beras Medium:')
    print(' '.join(baseline_cmd))
    subprocess.run(baseline_cmd, cwd=str(project_root), check=True)
    print('Recompute finished. Refresh the comparison cell to load updated results.')

Running baseline recompute for Beras Medium:
/home/rna_13/anaconda3/envs/rapids-24.10/bin/python /home/rna_13/DataScienceProject/dsproject_agriculturepredict/src/train.py --commodity Beras Medium --mode both --epochs 60 --device cpu --log-csv /home/rna_13/DataScienceProject/dsproject_agriculturepredict/runs/recomputed_metrics_Beras_Medium.csv --compare-csv /home/rna_13/DataScienceProject/dsproject_agriculturepredict/runs/recomputed_compare.csv --early-stopping-patience 5


2026-06-02 17:23:42.813906: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-06-02 17:23:42.827917: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:485] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2026-06-02 17:23:42.847154: E external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:8454] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
2026-06-02 17:23:42.852843: E external/local_xla/xla/stream_executor/cuda/cuda_blas.cc:1452] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2026-06-02 17:23:42.867222: I tensorflow/core/platform/cpu_feature_guar

Device: cpu
Epoch 1/60 - train_loss=0.781591 val_loss=0.927479
  Saved best model to models/lstm_Beras_Medium_univariate.pth
Epoch 2/60 - train_loss=0.213813 val_loss=0.119735
  Saved best model to models/lstm_Beras_Medium_univariate.pth
Epoch 3/60 - train_loss=0.076939 val_loss=0.029793
  Saved best model to models/lstm_Beras_Medium_univariate.pth
Epoch 4/60 - train_loss=0.046077 val_loss=0.030103
Epoch 5/60 - train_loss=0.035871 val_loss=0.033563
Epoch 6/60 - train_loss=0.029428 val_loss=0.038772
Epoch 7/60 - train_loss=0.024481 val_loss=0.045677
Epoch 8/60 - train_loss=0.019829 val_loss=0.036114
Early stopping after 8 epochs with no improvement for 5 epochs.
Loading best checkpoint from models/lstm_Beras_Medium_univariate.pth for final evaluation
Training finished in 3.2 sec

Final test metrics:
MAE: 125.748 Rp/kg
RMSE: 126.135 Rp/kg
MAPE: 1.006 %
sMAPE: 1.001 %
Directional Accuracy: 0.00 %
Device: cpu


/home/rna_13/DataScienceProject/dsproject_agriculturepredict/src/train.py:260: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  model.load_state_dict(torch.load(checkpoint_path

Epoch 1/60 - train_loss=0.443588 val_loss=0.170650
  Saved best model to models/lstm_Beras_Medium_multivariate.pth
Epoch 2/60 - train_loss=0.069214 val_loss=0.040356
  Saved best model to models/lstm_Beras_Medium_multivariate.pth
Epoch 3/60 - train_loss=0.034305 val_loss=0.024583
  Saved best model to models/lstm_Beras_Medium_multivariate.pth
Epoch 4/60 - train_loss=0.024771 val_loss=0.019724
  Saved best model to models/lstm_Beras_Medium_multivariate.pth
Epoch 5/60 - train_loss=0.018878 val_loss=0.021515
Epoch 6/60 - train_loss=0.015273 val_loss=0.022236
Epoch 7/60 - train_loss=0.016531 val_loss=0.018013
  Saved best model to models/lstm_Beras_Medium_multivariate.pth
Epoch 8/60 - train_loss=0.012886 val_loss=0.016527
  Saved best model to models/lstm_Beras_Medium_multivariate.pth
Epoch 9/60 - train_loss=0.012918 val_loss=0.017274
Epoch 10/60 - train_loss=0.012538 val_loss=0.018044
Epoch 11/60 - train_loss=0.009859 val_loss=0.015003
  Saved best model to models/lstm_Beras_Medium_multiv

/home/rna_13/DataScienceProject/dsproject_agriculturepredict/src/train.py:260: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  model.load_state_dict(torch.load(checkpoint_path

Recompute finished. Refresh the comparison cell to load updated results.


## Evaluation Insight

`runs/recomputed_compare.csv` sekarang adalah baseline yang digunakan untuk evaluasi. Ini adalah hasil terkini dari pelatihan ulang baseline menggunakan `src/train.py`, dan menggantikan potensi metrik lama yang tidak lagi konsisten dengan kondisi kode saat ini.

- `Beras Medium`: multivariat saat ini lebih baik dari univariat, dengan MAE sekitar 44.2 vs 70.4. Ini menandakan bahwa fitur produksi dan seasonalitas memberikan nilai tambah yang nyata untuk komoditas ini.
- `Kacang Hijau`: multivariat juga lebih kuat, dengan MAE sekitar 3.4 vs 9.8. Fitur eksogen tampaknya membantu stabilisasi prediksi harga kecil ini.
- `Jagung Pipil Kering`: baseline multivariat saat ini lebih buruk daripada univariat (MAE sekitar 117 vs 91), sehingga perlu ditinjau kembali fitur input atau konfigurasi LSTM untuk komoditas ini.
- `Beras Premium`: baseline multivariat saat ini jauh lebih buruk daripada univariat (MAE sekitar 465 vs 184), yang menunjukkan bahwa pipeline multivariat saat ini tidak terkalibrasi untuk komoditas ini.

Rekomendasi:
- Gunakan `runs/recomputed_compare.csv` sebagai baseline evaluasi saat ini.
- Fokuskan investigasi pada `Beras Premium` dan `Jagung Pipil Kering` untuk memperbaiki fitur multivariat atau target scaling.
- `Beras Medium` tetap prioritas untuk deployment lanjut karena multivariat memberikan peningkatan nyata di baseline terkini.
- Pastikan `src/train.py` di notebook memuat checkpoint terbaik sebelum evaluasi, dan gunakan `run_baseline = True` hanya jika Anda ingin merekomputasi ulang metrik dari notebook.


## Optuna Hyperparameter Tuning (opsional)

Sel ini akan menjalankan Optuna untuk komoditas yang direkomendasikan di atas. Atur `run_optuna=True` dan `trials` sesuai kebutuhan. Hati-hati: menjalankan Optuna memicu pelatihan beberapa kali dan membutuhkan waktu/compute.

In [51]:
import subprocess, sys
from pathlib import Path
run_optuna = True  # ubah ke True untuk menjalankan tuning
trials = 50
project_root = Path('..').resolve()

if run_optuna:
	if 'tune_recs' not in globals() or len(tune_recs) == 0:
		print('No recommended commodities found. Set `tune_list` manually, or run the previous cell to compute recommendations.')
		tune_list = []
	else:
		tune_list = tune_recs

	for commodity in tune_list:
		print('\n--- Running Optuna for', commodity, '---')
		cmd = [
			sys.executable,
			str(project_root / 'src' / 'train.py'),
			'--commodity', commodity,
			'--mode', 'multivariate',
			'--optuna',
			'--optuna-trials', str(trials),
			'--log-dir', str(project_root / f'runs/optuna/{commodity.replace(" ","_")}'),
			'--log-csv', str(project_root / 'runs' / 'optuna_metrics.csv'),
		]
		try:
			subprocess.run(cmd, cwd=str(project_root), check=True)
		except subprocess.CalledProcessError as e:
			print('Optuna run failed for', commodity, e)
else:
	print('Optuna tuning skipped (set run_optuna=True to enable)')


--- Running Optuna for Jagung Pipil Kering ---


2026-06-02 17:26:56.677857: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-06-02 17:26:56.692382: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:485] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2026-06-02 17:26:56.709045: E external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:8454] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
2026-06-02 17:26:56.713612: E external/local_xla/xla/stream_executor/cuda/cuda_blas.cc:1452] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2026-06-02 17:26:56.726061: I tensorflow/core/platform/cpu_feature_guar

Device: cuda
Epoch 1/10 - train_loss=1.180363 val_loss=0.564326
  Saved best model to models/lstm_Jagung_Pipil_Kering_multivariate.pth
Epoch 2/10 - train_loss=1.133637 val_loss=0.552243
  Saved best model to models/lstm_Jagung_Pipil_Kering_multivariate.pth
Epoch 3/10 - train_loss=1.092050 val_loss=0.539975
  Saved best model to models/lstm_Jagung_Pipil_Kering_multivariate.pth
Epoch 4/10 - train_loss=1.043296 val_loss=0.519588
  Saved best model to models/lstm_Jagung_Pipil_Kering_multivariate.pth
Epoch 5/10 - train_loss=0.994262 val_loss=0.494774
  Saved best model to models/lstm_Jagung_Pipil_Kering_multivariate.pth
Epoch 6/10 - train_loss=0.925604 val_loss=0.461417
  Saved best model to models/lstm_Jagung_Pipil_Kering_multivariate.pth
Epoch 7/10 - train_loss=0.839400 val_loss=0.418005
  Saved best model to models/lstm_Jagung_Pipil_Kering_multivariate.pth
Epoch 8/10 - train_loss=0.728629 val_loss=0.360454
  Saved best model to models/lstm_Jagung_Pipil_Kering_multivariate.pth
Epoch 9/10 

/home/rna_13/DataScienceProject/dsproject_agriculturepredict/src/train.py:260: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  model.load_state_dict(torch.load(checkpoint_path

Epoch 2/10 - train_loss=0.153787 val_loss=0.042464
  Saved best model to models/lstm_Jagung_Pipil_Kering_multivariate.pth
Epoch 3/10 - train_loss=0.091185 val_loss=0.052363
Epoch 4/10 - train_loss=0.063790 val_loss=0.049121
Epoch 5/10 - train_loss=0.060344 val_loss=0.040534
  Saved best model to models/lstm_Jagung_Pipil_Kering_multivariate.pth
Epoch 6/10 - train_loss=0.052188 val_loss=0.033173
  Saved best model to models/lstm_Jagung_Pipil_Kering_multivariate.pth
Epoch 7/10 - train_loss=0.053321 val_loss=0.030742
  Saved best model to models/lstm_Jagung_Pipil_Kering_multivariate.pth
Epoch 8/10 - train_loss=0.039711 val_loss=0.046194
Epoch 9/10 - train_loss=0.060069 val_loss=0.031242
Epoch 10/10 - train_loss=0.050014 val_loss=0.041938
Loading best checkpoint from models/lstm_Jagung_Pipil_Kering_multivariate.pth for final evaluation
Training finished in 0.8 sec

Final test metrics:
MAE: 13.266 Rp/kg
RMSE: 16.234 Rp/kg
MAPE: 0.340 %
sMAPE: 0.340 %
Directional Accuracy: 0.00 %
Device: cuda

...

/home/rna_13/DataScienceProject/dsproject_agriculturepredict/src/train.py:260: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  model.load_state_dict(torch.load(checkpoint_path

Epoch 1/10 - train_loss=0.125673 val_loss=0.158493
  Saved best model to models/lstm_Beras_Premium_multivariate.pth
Epoch 2/10 - train_loss=0.010114 val_loss=0.124593
  Saved best model to models/lstm_Beras_Premium_multivariate.pth
Epoch 3/10 - train_loss=0.005056 val_loss=0.122832
  Saved best model to models/lstm_Beras_Premium_multivariate.pth
Epoch 4/10 - train_loss=0.003796 val_loss=0.119424
  Saved best model to models/lstm_Beras_Premium_multivariate.pth
Epoch 5/10 - train_loss=0.003595 val_loss=0.120403
Epoch 6/10 - train_loss=0.003279 val_loss=0.124075
Epoch 7/10 - train_loss=0.003045 val_loss=0.117436
  Saved best model to models/lstm_Beras_Premium_multivariate.pth
Epoch 8/10 - train_loss=0.003223 val_loss=0.128746
Epoch 9/10 - train_loss=0.002778 val_loss=0.110275
  Saved best model to models/lstm_Beras_Premium_multivariate.pth
Epoch 10/10 - train_loss=0.003036 val_loss=0.111739
Loading best checkpoint from models/lstm_Beras_Premium_multivariate.pth for final evaluation
Traini

/home/rna_13/DataScienceProject/dsproject_agriculturepredict/src/train.py:260: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  model.load_state_dict(torch.load(checkpoint_path

Epoch 1/60 - train_loss=0.064884 val_loss=0.162301
  Saved best model to /home/rna_13/DataScienceProject/dsproject_agriculturepredict/runs/optuna/Beras_Premium/lstm_Beras_Premium_multivariate.pth
Epoch 2/60 - train_loss=0.007414 val_loss=0.154441
  Saved best model to /home/rna_13/DataScienceProject/dsproject_agriculturepredict/runs/optuna/Beras_Premium/lstm_Beras_Premium_multivariate.pth
Epoch 3/60 - train_loss=0.004193 val_loss=0.151934
  Saved best model to /home/rna_13/DataScienceProject/dsproject_agriculturepredict/runs/optuna/Beras_Premium/lstm_Beras_Premium_multivariate.pth
Epoch 4/60 - train_loss=0.003374 val_loss=0.163729
Epoch 5/60 - train_loss=0.003178 val_loss=0.145627
  Saved best model to /home/rna_13/DataScienceProject/dsproject_agriculturepredict/runs/optuna/Beras_Premium/lstm_Beras_Premium_multivariate.pth
Epoch 6/60 - train_loss=0.002497 val_loss=0.145014
  Saved best model to /home/rna_13/DataScienceProject/dsproject_agriculturepredict/runs/optuna/Beras_Premium/lstm_

/home/rna_13/DataScienceProject/dsproject_agriculturepredict/src/train.py:260: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  model.load_state_dict(torch.load(checkpoint_path

### Insight Ringkas dari Hasil Optuna

- **Ringkasan umum:** Optuna sudah menjalankan retrain multivariate untuk semua komoditas. Saya membandingkan metrik multivariate Optuna (`runs/all_commodity_optuna_summary.csv`) dengan baseline multivariate saat ini (`runs/recomputed_compare.csv`) dan menyimpan ringkasan di `runs/optuna_vs_baseline_summary.csv`.
- **Hasil per komoditas (MAE baseline → MAE Optuna):**
    - Beras Premium: 465.15 → 288.05 (peningkatan besar; gunakan model Optuna sebagai kandidat utama)
    - Jagung Pipil Kering: 117.20 → 78.81 (peningkatan jelas; layak dipakai)
    - Kacang Hijau: 3.38 → 3.19 (peningkatan kecil; stabil)
    - Beras Medium: 64.40 → 132.45 (Optuna MUNTAH; hasil Optuna lebih buruk — investigasi diperlukan)
- **Rekomendasi tindakan singkat:**
    1. Teruskan dan gunakan model Optuna untuk `Beras Premium` dan `Jagung Pipil Kering` (evaluasi kembali pada holdout/horizon operasional sebelum deployment).
    2. Untuk `Beras Medium`, cek data input untuk Optuna retrain (scaling, join eksogen, horizon), dan ulangi Optuna dengan batasan/hyperprior yang lebih konservatif atau kurangi search space.
    3. `Kacang Hijau` menunjukkan sedikit perbaikan — pertimbangkan run Optuna tambahan dengan fokus pada stability/regularization jika biaya compute kecil.
    4. Simpan model terbaik Optuna + checkpoint dan tambahkan kolom per-commodity `optuna_source` pada katalog eksperimen untuk traceability.

Jika mau, saya bisa: memeriksa `runs/optuna_metrics.csv` untuk melihat hyperparameter terpilih, menyusun per-commodity plot per-trial, atau menyiapkan skrip untuk mempromosikan model Optuna terpilih ke folder `models/optuna/`.